## Understanidng what StatsBomb contains

In [ ]:
from statsbombpy import sb
import pandas as pd

competitions = sb.competitions()
competitions.head()

In [12]:
competitions['competition_name'].unique()

array(['1. Bundesliga', 'African Cup of Nations', 'Champions League',
       'Copa America', 'Copa del Rey', "FA Women's Super League",
       'FIFA U20 World Cup', 'FIFA World Cup', 'Frauen Bundesliga',
       'Indian Super league', 'La Liga', 'Liga F', 'Liga Profesional',
       'Ligue 1', 'Major League Soccer', 'North American League', 'NWSL',
       'Premier League', 'Serie A', 'Serie A Women', 'UEFA Euro',
       'UEFA Europa League', "UEFA Women's Euro", "Women's World Cup"],
      dtype=object)

In [15]:
pl = competitions[competitions['competition_name'] == 'Premier League']
pl.head()

,competition_id,season_id,country_name,competition_name,competition_gender,competition_youth,competition_international,season_name,match_updated,match_updated_360,match_available_360,match_available
68,2,27,England,Premier League,male,False,False,2015/2016,2025-12-17T14:38:00.447595,2021-06-13T16:17:31.694,None,2025-12-17T14:38:00.447595
69,2,44,England,Premier League,male,False,False,2003/2004,2025-06-24T13:53:07.585114,2021-06-13T16:17:31.694,None,2025-06-24T13:53:07.585114


In [ ]:
matches = sb.matches(competition_id=2, season_id=27)
matches.head()

In [ ]:
matches.columns.tolist()

In [27]:
matches[['match_id','home_team', 'away_team', 'home_score', 'away_score', 'match_date', 'match_week', 'home_managers', 'away_managers']].head()

,match_id,home_team,away_team,home_score,away_score,match_date,match_week,home_managers,away_managers
0,3754217,Chelsea,Arsenal,2,0,2015-09-19,6,José Mario Felix dos Santos Mourinho,Arsène Wenger
1,3754117,Aston Villa,Arsenal,0,2,2015-12-13,16,Rémi Garde,Arsène Wenger
2,3754296,Arsenal,Manchester City,2,1,2015-12-21,17,Arsène Wenger,Manuel Luis Pellegrini Ripamonti
3,3753983,Swansea City,Arsenal,0,3,2015-10-31,11,Garry Monk,Arsène Wenger
4,3754160,Arsenal,Sunderland,3,1,2015-12-05,15,Arsène Wenger,Sam Allardyce


In [ ]:
events = sb.events(match_id=3754217)
events.head()

In [ ]:
events.columns.tolist()

### Preparing the passing networks for analysis and plotting

In [30]:
events[events['type'] == 'Pass'][['player', 'pass_recipient', 'location', 'pass_end_location', 'index']].head(10)

,player,pass_recipient,location,pass_end_location,index
6,Theo Walcott,Mesut Özil,"[61.0, 40.1]","[59.1, 42.1]",5
7,Mesut Özil,Santiago Cazorla González,"[59.3, 42.1]","[43.7, 38.2]",7
8,Santiago Cazorla González,Aaron Ramsey,"[49.5, 41.7]","[63.1, 48.9]",10
9,Aaron Ramsey,Theo Walcott,"[65.9, 48.7]","[69.6, 48.9]",14
10,Francesc Fàbregas i Soler,Kurt Happy Zouma,"[47.1, 38.2]","[32.6, 39.3]",19
11,Kurt Happy Zouma,Pedro Eliezer Rodríguez Ledesma,"[34.3, 44.2]","[51.0, 51.7]",22
12,Aaron Ramsey,NaN,"[78.8, 32.0]","[76.0, 33.1]",28
13,Mesut Özil,Theo Walcott,"[76.4, 39.5]","[83.5, 45.1]",31
14,Nemanja Matić,Asmir Begović,"[33.4, 36.5]","[6.9, 42.9]",35
15,Asmir Begović,NaN,"[9.3, 46.8]","[69.8, 63.9]",38


#### NaNs could mean a pass went out of play or a clearance

In [31]:
events[events['type'] == 'Pass'][['player', 'pass_recipient', 'location', 'pass_end_location', 'index']].info()

<class 'pandas.core.frame.DataFrame'>
Index: 1046 entries, 6 to 1051
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   player             1046 non-null   object
 1   pass_recipient     963 non-null    object
 2   location           1046 non-null   object
 3   pass_end_location  1046 non-null   object
 4   index              1046 non-null   int64 
dtypes: int64(1), object(4)
memory usage: 49.0+ KB


### For all matches we must take all 380 games from the season

In [ ]:
all_events = []
for match_id in matches['match_id']:
    events = sb.events(match_id=match_id)
    events['match_id'] = match_id
    all_events.append(events)

all_events_df = pd.concat(all_events, ignore_index=True)

In [33]:
all_events_df.shape

(1313773, 118)

In [44]:
all_events_df.to_parquet('../data/all_events_2015_16.parquet', engine='fastparquet')

### We have a dataset we can work with!!